# ⚡ VRP Factor Model — Real-Time (TradingView Feed)

Pulls live tick data via TradingView's WebSocket. No API key needed.

**Just run Cell 1 then Cell 2 — everything else is automatic.**

```
VRP = IV² − RV²
```
VRP updates every minute using fresh tick data from TradingView.

In [1]:
# ── CELL 1 · Install everything ────────────────────────────────────────────
import subprocess, sys

pkgs = [
    'git+https://github.com/StreamAlpha/tvdatafeed.git',
    'pandas', 'numpy', 'matplotlib', 'scipy',
    'statsmodels', 'ipywidgets'
]

for pkg in pkgs:
    print(f'Installing {pkg.split("/")[-1]}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  ⚠️  {result.stderr.strip()[-200:]}')
    else:
        print(f'  ✓')

print('\n✅ All done — run Cell 2 to start the live feed')

Installing tvdatafeed.git...
  ⚠️  1 did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Installing pandas...
  ✓
Installing numpy...
  ✓
Installing matplotlib...
  ✓
Installing scipy...
  ✓
Installing statsmodels...
  ✓
Installing ipywidgets...
  ✓

✅ All done — run Cell 2 to start the live feed


In [2]:
# ── CELL 2 · Imports & config ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
import warnings, time, threading
from datetime import datetime
warnings.filterwarnings('ignore')

from tvdatafeed import TvDatafeed, Interval

# ── Tickers to track (TradingView symbol, exchange) ──
SYMBOLS = [
    ('AAPL',  'NASDAQ'),
    ('MSFT',  'NASDAQ'),
    ('NVDA',  'NASDAQ'),
    ('GOOGL', 'NASDAQ'),
    ('META',  'NASDAQ'),
    ('AMZN',  'NASDAQ'),
    ('TSLA',  'NASDAQ'),
    ('JPM',   'NYSE'),
    ('BAC',   'NYSE'),
    ('GS',    'NYSE'),
    ('XOM',   'NYSE'),
    ('JNJ',   'NYSE'),
    ('UNH',   'NYSE'),
    ('WMT',   'NYSE'),
    ('HD',    'NYSE'),
]

RV_WINDOW  = 21    # bars for realised variance
IV_WINDOW  = 5     # bars for implied variance proxy
REFRESH_S  = 60    # seconds between data refreshes

# Connect — no login needed for free delayed data
print('Connecting to TradingView...')
tv = TvDatafeed()
print('✓ Connected')

ModuleNotFoundError: No module named 'tvdatafeed'

In [ ]:
# ── CELL 3 · Core VRP functions ────────────────────────────────────────────

def fetch_bars(symbol, exchange, n_bars=100, interval=Interval.in_daily):
    """Pull last n_bars candles for a symbol from TradingView."""
    try:
        df = tv.get_hist(symbol=symbol, exchange=exchange,
                         interval=interval, n_bars=n_bars)
        if df is None or len(df) < RV_WINDOW + 5:
            return None
        df = df[['open','high','low','close','volume']].copy()
        df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
        return df
    except Exception as e:
        print(f'  ⚠️  {symbol}: {e}')
        return None


def compute_rv2(df):
    """Realised variance: rolling sum of squared log returns (annualised)."""
    rv2 = (df['log_ret']**2).rolling(RV_WINDOW).sum() * (252 / RV_WINDOW)
    return float(rv2.iloc[-1])


def compute_iv2_proxy(df):
    """
    Implied variance proxy via Parkinson (1980) high-low estimator.
    IV² = (1/4ln2) × mean[(ln(H/L))²] × 252
    Better than close-to-close for intraday vol, no options data needed.
    """
    hl = np.log(df['high'] / df['low'])
    iv2 = ((hl**2) / (4 * np.log(2))).rolling(IV_WINDOW).mean() * 252
    return float(iv2.iloc[-1])


def compute_vrp(df):
    """VRP = IV² − RV² for a single symbol."""
    iv2 = compute_iv2_proxy(df)
    rv2 = compute_rv2(df)
    return {'iv2': iv2, 'rv2': rv2, 'vrp': iv2 - rv2,
            'price': float(df['close'].iloc[-1]),
            'last_bar': df.index[-1]}


def fetch_all_vrp():
    """Pull data for all symbols and compute VRP cross-section."""
    rows = []
    for sym, exch in SYMBOLS:
        df = fetch_bars(sym, exch, n_bars=150)
        if df is not None:
            result = compute_vrp(df)
            result['symbol'] = sym
            rows.append(result)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index('symbol')


def fama_macbeth_slope(vrp_series, ret_series):
    """Simple single-period FM slope of returns on VRP."""
    df = pd.DataFrame({'vrp': vrp_series, 'ret': ret_series}).dropna()
    if len(df) < 5:
        return np.nan
    X = np.column_stack([np.ones(len(df)), df['vrp']])
    coefs, *_ = np.linalg.lstsq(X, df['ret'].values, rcond=None)
    return coefs[1]


print('✓ VRP functions ready')

In [ ]:
# ── CELL 4 · Initial data pull ─────────────────────────────────────────────
print(f'Fetching data for {len(SYMBOLS)} symbols...')
vrp_now = fetch_all_vrp()

if vrp_now.empty:
    print('❌ No data returned. Check your internet connection.')
else:
    vrp_now_sorted = vrp_now.sort_values('vrp', ascending=False)
    print(f'\n✓ Live VRP snapshot — {datetime.now().strftime("%H:%M:%S")}')
    print(f'{"Symbol":<8} {"Price":>8} {"IV²":>8} {"RV²":>8} {"VRP":>8}  Signal')
    print('─' * 58)
    for sym, row in vrp_now_sorted.iterrows():
        signal = '🟢 LONG ' if row['vrp'] > 0.01 else ('🔴 SHORT' if row['vrp'] < -0.01 else '⚪ FLAT ')
        print(f'{sym:<8} {row["price"]:>8.2f} {row["iv2"]:>8.4f} {row["rv2"]:>8.4f} {row["vrp"]:>+8.4f}  {signal}')
    print(f'\nMean VRP: {vrp_now["vrp"].mean():+.4f}  |  '
          f'% positive: {(vrp_now["vrp"]>0).mean()*100:.0f}%')

In [ ]:
# ── CELL 5 · Live auto-refreshing dashboard ────────────────────────────────
#
# Runs a live matplotlib dashboard that refreshes every REFRESH_S seconds.
# Press the ■ Stop button in Jupyter to stop it.
#
from IPython.display import clear_output

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F8F8F6',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.5,
    'font.family': 'monospace', 'font.size': 10,
})

# History buffers
history_time = []
history_mean_vrp = []
history_pct_pos  = []
history_snapshots = []   # list of full vrp DataFrames

def refresh_and_plot():
    global vrp_now
    vrp_now = fetch_all_vrp()
    if vrp_now.empty:
        print('No data')
        return

    now = datetime.now()
    history_time.append(now)
    history_mean_vrp.append(vrp_now['vrp'].mean())
    history_pct_pos.append((vrp_now['vrp'] > 0).mean() * 100)
    history_snapshots.append(vrp_now.copy())

    clear_output(wait=True)

    fig = plt.figure(figsize=(14, 8))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── Top: VRP bar chart (current cross-section) ──
    ax1 = fig.add_subplot(gs[0, :])
    vrp_sorted = vrp_now.sort_values('vrp')
    colors = ['#3B6D11' if v > 0 else '#A32D2D' for v in vrp_sorted['vrp']]
    bars = ax1.barh(vrp_sorted.index, vrp_sorted['vrp'], color=colors, height=0.6)
    ax1.axvline(0, color='#444441', linewidth=0.8)
    ax1.set_title(f'Live VRP Cross-Section  ·  {now.strftime("%Y-%m-%d  %H:%M:%S")}',
                  fontsize=11)
    ax1.set_xlabel('VRP = IV² − RV²  (annualised variance)')
    for bar, (sym, row) in zip(bars, vrp_sorted.iterrows()):
        ax1.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
                 f'${row["price"]:.0f}', va='center', fontsize=8,
                 color='#5F5E5A')

    # ── Bottom-left: mean VRP over time ──
    ax2 = fig.add_subplot(gs[1, 0])
    if len(history_time) > 1:
        ax2.plot(history_time, history_mean_vrp,
                 color='#185FA5', linewidth=1.5, marker='o', markersize=3)
        ax2.fill_between(history_time, 0, history_mean_vrp,
                         where=[v>=0 for v in history_mean_vrp],
                         color='#3B6D11', alpha=0.2)
        ax2.fill_between(history_time, 0, history_mean_vrp,
                         where=[v<0 for v in history_mean_vrp],
                         color='#A32D2D', alpha=0.2)
        ax2.axhline(0, color='#444441', linewidth=0.6)
    ax2.set_title('Mean VRP (session)')
    ax2.set_ylabel('VRP')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

    # ── Bottom-middle: % stocks with positive VRP ──
    ax3 = fig.add_subplot(gs[1, 1])
    if len(history_time) > 1:
        ax3.plot(history_time, history_pct_pos,
                 color='#BA7517', linewidth=1.5, marker='o', markersize=3)
        ax3.axhline(50, color='#444441', linewidth=0.6, linestyle='--')
        ax3.set_ylim(0, 100)
    ax3.set_title('% stocks positive VRP')
    ax3.set_ylabel('%')
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

    # ── Bottom-right: IV² vs RV² scatter ──
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.scatter(vrp_now['rv2'], vrp_now['iv2'],
                color='#185FA5', alpha=0.8, s=50, zorder=3)
    for sym, row in vrp_now.iterrows():
        ax4.annotate(sym, (row['rv2'], row['iv2']),
                     fontsize=7, color='#5F5E5A',
                     xytext=(3, 3), textcoords='offset points')
    lim_max = max(vrp_now[['iv2','rv2']].max().max() * 1.1, 0.01)
    ax4.plot([0, lim_max], [0, lim_max], color='#A32D2D',
             linewidth=0.8, linestyle='--', label='IV²=RV² (VRP=0)')
    ax4.set_xlabel('RV² (realised)')
    ax4.set_ylabel('IV² (implied)')
    ax4.set_title('IV² vs RV²')
    ax4.legend(fontsize=8)

    plt.suptitle('VRP Factor Model — Live TradingView Feed', fontsize=12, y=1.01)
    plt.savefig('vrp_live.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Print signal table ──
    vrp_s = vrp_now.sort_values('vrp', ascending=False)
    print(f'\n{"Symbol":<8} {"VRP":>8}  Signal')
    print('─'*30)
    for sym, row in vrp_s.iterrows():
        sig = '🟢 LONG ' if row['vrp'] > 0.01 else ('🔴 SHORT' if row['vrp'] < -0.01 else '⚪ FLAT ')
        print(f'{sym:<8} {row["vrp"]:>+8.4f}  {sig}')
    print(f'\nNext refresh in {REFRESH_S}s  ·  Press ■ Stop to quit')


# ── Live loop ──
print('Starting live dashboard... (runs until you press ■ Stop in Jupyter)')
try:
    while True:
        refresh_and_plot()
        time.sleep(REFRESH_S)
except KeyboardInterrupt:
    print('\n⏹ Stopped.')

In [ ]:
# ── CELL 6 · One-shot snapshot (use this if Cell 5 feels too busy) ─────────
# Just run this cell any time you want a fresh snapshot without the live loop.

print('Fetching fresh snapshot...')
snap = fetch_all_vrp()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
snap_s = snap.sort_values('vrp')
colors = ['#3B6D11' if v > 0 else '#A32D2D' for v in snap_s['vrp']]
axes[0].barh(snap_s.index, snap_s['vrp'], color=colors, height=0.6)
axes[0].axvline(0, color='#444441', linewidth=0.8)
axes[0].set_title(f'VRP Cross-Section  ·  {datetime.now().strftime("%H:%M:%S")}')
axes[0].set_xlabel('VRP = IV² − RV²')

# Scatter
axes[1].scatter(snap['rv2'], snap['iv2'], color='#185FA5', s=60, zorder=3)
for sym, row in snap.iterrows():
    axes[1].annotate(sym, (row['rv2'], row['iv2']),
                     fontsize=8, xytext=(3,3), textcoords='offset points')
lim = max(snap[['iv2','rv2']].max().max() * 1.1, 0.01)
axes[1].plot([0,lim],[0,lim], color='#A32D2D', linewidth=0.8,
             linestyle='--', label='VRP = 0')
axes[1].set_xlabel('RV²'); axes[1].set_ylabel('IV²')
axes[1].set_title('IV² vs RV² scatter'); axes[1].legend()

plt.tight_layout()
plt.savefig('vrp_snapshot.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop 3 LONG (highest VRP):')
print(snap.sort_values('vrp', ascending=False).head(3)[['price','iv2','rv2','vrp']].round(4).to_string())
print(f'\nTop 3 SHORT (lowest VRP):')
print(snap.sort_values('vrp').head(3)[['price','iv2','rv2','vrp']].round(4).to_string())